# 📰 A Real-World Agent from Scratch: Hacker News Chatbot

## Learning Objectives
In this notebook, you will learn:
1. **The Architecture** - A ReAct agent that chooses between three Hacker News tools
2. **Packaging the Agent Code** - How the classes from notebooks 01–03 live on as a small library, `agentic_patterns`
3. **Real Tools** - Wrap the public Hacker News API (stories, comments, page content) with the `tool` decorator
4. **Running the Bot** - Assemble the agent and watch it answer single- and multi-tool questions
5. **A Streamlit UI** - Write a chat app around the same agent and launch it

## Where this fits
`01_reflection_pattern`, `02_tool_use_pattern` and `03_planning_pattern_react` built each pattern inline. This notebook imports the same classes from a package and builds something you could actually use. Module `02_tools_and_function_calling` then replaces the hand-written tool plumbing with the model's native tool calling.

## Prerequisites
- The `tool` decorator (`02_tool_use_pattern`) and the `ReactAgent` loop (`03_planning_pattern_react`)
- Dependencies installed with `uv sync` at the repository root. Launching the UI in Part 5 also needs the optional group: `uv sync --group apps`
- `OPENAI_API_KEY` in the project-root `.env`. The Hacker News API is public and needs no key; the tools call it live.

---
## 🏗️ The Architecture

<img src="../assets/img/real_world_project.png" alt="Hacker News chatbot architecture" width="500"/>

---

In this final notebook of the module, we implement the architecture you can see above. It's a ReAct Agent that will choose between three tools (**StoriesTool**, **CommentsTool** and **ContentTool**) to solve the user task.

In addition, we'll implement a Streamlit interface, that will enable us to chat with our agent.

---
## 📦 Step 1: Environment Setup

Load `OPENAI_API_KEY` from the project-root `.env` and pick the model. There is no client to create here: the agent builds its own raw OpenAI client.

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

# The agent creates its own raw OpenAI client; this is the model it will use
MODEL = "gpt-4o-mini"

print(f"✅ Environment ready, model: {MODEL}")

✅ Environment ready, model: gpt-4o-mini


---
## 📚 Part 1: The Agent Code as a Library

Notebooks 01–03 defined `tool`, `Tool`, `ToolAgent`, `ReactAgent`, `ReflectionAgent` and their helpers inline. An app can't import from a notebook, so the same code is packaged as a Python package, **`agentic_patterns`**, in the `agentic_patterns/` folder beside this notebook (so `import agentic_patterns` works from here). It comes from the open-source, MIT-licensed [neural-maze/agentic_patterns](https://github.com/neural-maze/agentic_patterns) project, adapted for this course to use the OpenAI SDK with `gpt-4o-mini`.

| Module | What it holds | Built in |
|---|---|---|
| `utils/completions.py` | `completions_create`, prompt builders, `ChatHistory`, `FixedFirstChatHistory` | `01_reflection_pattern` |
| `utils/extraction.py` | `extract_tag_content` | `02_tool_use_pattern` |
| `reflection_pattern/reflection_agent.py` | `ReflectionAgent` | `01_reflection_pattern` |
| `tool_pattern/tool.py` | `get_fn_signature`, `validate_arguments`, `Tool`, `tool` | `02_tool_use_pattern` |
| `tool_pattern/tool_agent.py` | `ToolAgent` | `02_tool_use_pattern` |
| `planning_pattern/react_agent.py` | `ReactAgent` | `03_planning_pattern_react` |

In [2]:
# ============================================================================
# LIBRARY TOUR: the package layout and the ReactAgent signature
# ============================================================================
import inspect
from pathlib import Path

import agentic_patterns
from agentic_patterns.planning_pattern.react_agent import ReactAgent

lib_dir = Path(agentic_patterns.__file__).parent
print("📁 agentic_patterns/")
for path in sorted(lib_dir.rglob("*.py")):
    print("   ", path.relative_to(lib_dir))

print("\nReactAgent" + str(inspect.signature(ReactAgent)))

📁 agentic_patterns/
    __init__.py
    planning_pattern/__init__.py
    planning_pattern/react_agent.py
    reflection_pattern/__init__.py
    reflection_pattern/reflection_agent.py
    tool_pattern/__init__.py
    tool_pattern/tool.py
    tool_pattern/tool_agent.py
    utils/__init__.py
    utils/completions.py
    utils/extraction.py
    utils/logging.py

ReactAgent(tools: agentic_patterns.tool_pattern.tool.Tool | list[agentic_patterns.tool_pattern.tool.Tool], model: str = 'gpt-4o-mini', system_prompt: str = '', api_key: str | None = None) -> None


The constructor is the one you wrote in `03_planning_pattern_react`, plus an optional `api_key` (the Streamlit app could pass one; `None` reads `OPENAI_API_KEY` from the environment).

---
## 🛠️ Part 2: The Hacker News Tools

The bot gets three tools: **StoriesTool** (`get_hn_stories`), **CommentsTool** (`get_relevant_comments`) and **ContentTool** (`get_story_content`). We write them to `hn_tools.py` beside this notebook so the Streamlit app can import them too. Three helper functions (`fetch_item`, `fetch_story_ids`, `fetch_text`) are plain functions, not tools: the model never sees them.

Two guards keep the agent's context small: comments are limited to the first `limit` top-level comments, and page text is cut to 4,000 characters.

In [3]:
%%writefile hn_tools.py
# ============================================================================
# HACKER NEWS TOOLS: three @tool functions over the public Firebase API
# ============================================================================
import json

import requests
from bs4 import BeautifulSoup

from agentic_patterns.tool_pattern.tool import tool

BASE_URL = "https://hacker-news.firebaseio.com/v0"
MAX_STORY_CHARS = 4000  # keep page text small enough for the model's context


def fetch_item(item_id: int):
    """
    Fetches details of a story by its ID.

    Args:
        item_id (int): The ID of the item to fetch.

    Returns:
        dict: Details of the story.
    """
    url = f"{BASE_URL}/item/{item_id}.json"
    response = requests.get(url, timeout=10)
    return response.json() or {}


def fetch_story_ids(story_type: str = "top", limit: int = None):
    """
    Fetches the top story IDs.

    Args:
        story_type: The story type. Defaults to top (`topstories.json`)
        limit: The limit of stories to be fetched.

    Returns:
        List[int]: A list of top story IDs.
    """
    url = f"{BASE_URL}/{story_type}stories.json"
    response = requests.get(url, timeout=10)
    story_ids = response.json()

    if limit:
        story_ids = story_ids[:limit]

    return story_ids


def fetch_text(url: str):
    """
    Fetches the text from a URL (if there's text to be fetched). If it fails,
    it will return an informative message to the LLM.

    Args:
        url: The story URL

    Returns:
        A string representing whether the story text or an informative error (represented as a string)
    """
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, "html.parser")
            text_content = " ".join(soup.get_text().split())
            return text_content[:MAX_STORY_CHARS]
        else:
            return f"Unable to fetch content from {url}. Status code: {response.status_code}"
    except Exception as e:
        return f"An error occurred: {e}"


@tool
def get_hn_stories(limit: int = 5, story_type: str = "top"):
    """
    Fetches the top Hacker News stories based on the provided parameters.

    Args:
        limit (int): The number of top stories to retrieve. Default is 5.
        story_type (str): The story type: "top", "new" or "best". Default is "top".

    Returns:
        list[Dict[str, Union[str, int]]]: A list of dictionaries containing
        'story_id', 'title', 'url', and 'score' of the stories.
    """
    story_ids = fetch_story_ids(story_type, limit)
    stories = [fetch_item(story_id) for story_id in story_ids]

    formatted_stories = []
    for story in stories:
        formatted_stories.append({
            "title": story.get("title"),
            "url": story.get("url"),
            "score": story.get("score"),
            "story_id": story.get("id"),
        })

    return formatted_stories[:limit]


@tool
def get_relevant_comments(story_id: int, limit: int = 10):
    """
    Get the most relevant comments for a Hacker News item.

    Args:
        story_id: The ID of the Hacker News item.
        limit: The number of comments to retrieve (default is 10).

    Returns:
        A list of comment texts.
    """
    story = fetch_item(story_id)

    if "kids" not in story:
        return "This item doesn't have comments."

    # Hacker News already returns top-level comments ('kids') in ranked order,
    # so the first `limit` are the most relevant ones.
    comments = [fetch_item(cid) for cid in story["kids"][:limit]]
    return json.dumps([c["text"] for c in comments if c.get("text")])


@tool
def get_story_content(story_url: str):
    """
    Gets the content of the story.

    Args:
        story_url: A string representing the story URL

    Returns:
        The content of the story
    """
    return fetch_text(story_url)

Overwriting hn_tools.py


Before handing the tools to an agent, call one directly. A `Tool` runs its function with `.run(**kwargs)`:

In [4]:
# ============================================================================
# TRY A TOOL: the current top 3 stories
# ============================================================================
import json

from hn_tools import get_hn_stories, get_relevant_comments, get_story_content

print("Tools:", [t.name for t in (get_hn_stories, get_relevant_comments, get_story_content)])
print(json.dumps(get_hn_stories.run(limit=3), indent=2))

Tools: ['get_hn_stories', 'get_relevant_comments', 'get_story_content']


[
  {
    "title": "Pi.dev: You Said No MCP",
    "url": "https://earendil.com/posts/you-said-no-mcp/",
    "score": 191,
    "story_id": 49906637
  },
  {
    "title": "Show HN: JBR-001 \u2013 An open-source 3D printable desktop robot",
    "url": "https://projecthub.arduino.cc/syntheticaidata/jbr-001-a-desktop-companion-robot-powered-by-arduino-uno-q-b11c96",
    "score": 35,
    "story_id": 49890707
  },
  {
    "title": "Livenerf: Has Opus 5.5 been nerfed yet?",
    "url": "https://github.com/ninjahawk/livenerf",
    "score": 701,
    "story_id": 49901736
  }
]


---
## 🤖 Part 3: Creating the ReAct Agent

`hn_bot.py` builds a `ReactAgent` with the three tools and a system prompt that sets the bot's persona and asks for stories as a markdown table. It is a file (not just a cell) because the Streamlit app imports `get_hn_bot` from it.

In [5]:
%%writefile hn_bot.py
# ============================================================================
# HN BOT: a ReactAgent from agentic_patterns wired to the three HN tools
# ============================================================================
from agentic_patterns.planning_pattern.react_agent import ReactAgent
from hn_tools import get_hn_stories, get_relevant_comments, get_story_content

BOT_SYSTEM_PROMPT = """You are the Singularity Incarnation of Hacker News.
The human will ask you for information about Hacker News.
If you can't find any information about the question asked
or the result is incomplete, apologise to the human and ask him if
you can help him with something else.
If the human asks you to show him stories, do it using a markdown table.
The markdown table has the following format:

story_id | title | url | score"""


def get_hn_bot(model: str = "gpt-4o-mini", api_key: str | None = None) -> ReactAgent:
    """Build the Hacker News ReAct agent. api_key=None reads OPENAI_API_KEY from the environment."""
    return ReactAgent(
        system_prompt=BOT_SYSTEM_PROMPT,
        tools=[get_hn_stories, get_relevant_comments, get_story_content],
        model=model,
        api_key=api_key,
    )

Overwriting hn_bot.py


---
## 🚀 Part 4: Running the Bot

First a question that needs one tool call, then one that needs two: the agent has to find the top story's id before it can fetch that story's comments. The live answers change with Hacker News.

In [6]:
# ============================================================================
# RUN: one tool call
# ============================================================================
from IPython.display import display_markdown

from hn_bot import get_hn_bot

bot = get_hn_bot(model=MODEL)

answer = bot.run("Show me the top 5 Hacker News stories right now")
display_markdown(answer, raw=True)


Thought: I need to fetch the top Hacker News stories with a limit of 5.

Using Tool: get_hn_stories

Tool call dict: 
{'name': 'get_hn_stories', 'arguments': {'limit': 5, 'story_type': 'top'}, 'id': 0}



Tool result: 
[{'title': 'Pi.dev: You Said No MCP', 'url': 'https://earendil.com/posts/you-said-no-mcp/', 'score': 191, 'story_id': 49906637}, {'title': 'Show HN: JBR-001 – An open-source 3D printable desktop robot', 'url': 'https://projecthub.arduino.cc/syntheticaidata/jbr-001-a-desktop-companion-robot-powered-by-arduino-uno-q-b11c96', 'score': 35, 'story_id': 49890707}, {'title': 'Livenerf: Has Opus 5.5 been nerfed yet?', 'url': 'https://github.com/ninjahawk/livenerf', 'score': 701, 'story_id': 49901736}, {'title': 'Solving Factorio Quality', 'url': 'https://exyr.org/2026/solving-factorio-quality/', 'score': 134, 'story_id': 49887343}, {'title': 'Most data centers refusing to say how much water, electricity they use', 'url': 'https://nltimes.nl/2026/09/30/data-centers-refusing-say-much-water-electricity-use', 'score': 54, 'story_id': 49907057}]

Observations: {0: [{'title': 'Pi.dev: You Said No MCP', 'url': 'https://earendil.com/posts/you-said-no-mcp/', 'score': 191, 'story_id': 499

Here are the top 5 Hacker News stories right now:

| story_id | title                                                                  | url                                                                                           | score |
|----------|------------------------------------------------------------------------|-----------------------------------------------------------------------------------------------|-------|
| 49906637 | Pi.dev: You Said No MCP                                               | [Read More](https://earendil.com/posts/you-said-no-mcp/)                                  | 191   |
| 49890707 | Show HN: JBR-001 – An open-source 3D printable desktop robot          | [Read More](https://projecthub.arduino.cc/syntheticaidata/jbr-001-a-desktop-companion-robot-powered-by-arduino-uno-q-b11c96) | 35    |
| 49901736 | Livenerf: Has Opus 5.5 been nerfed yet?                               | [Read More](https://github.com/ninjahawk/livenerf)                                         | 701   |
| 49887343 | Solving Factorio Quality                                              | [Read More](https://exyr.org/2026/solving-factorio-quality/)                               | 134   |
| 49907057 | Most data centers refusing to say how much water, electricity they use| [Read More](https://nltimes.nl/2026/09/30/data-centers-refusing-say-much-water-electricity-use) | 54    |

In [7]:
# ============================================================================
# RUN: two dependent tool calls
# ============================================================================
answer = bot.run("What is the current top story on Hacker News, and what are people saying about it in the first 5 comments?")
display_markdown(answer, raw=True)


Thought: I need to fetch the current top stories from Hacker News to identify the top story and its comments.

Using Tool: get_hn_stories

Tool call dict: 
{'name': 'get_hn_stories', 'arguments': {'limit': 1, 'story_type': 'top'}, 'id': 0}



Tool result: 
[{'title': 'Pi.dev: You Said No MCP', 'url': 'https://earendil.com/posts/you-said-no-mcp/', 'score': 191, 'story_id': 49906637}]

Observations: {0: [{'title': 'Pi.dev: You Said No MCP', 'url': 'https://earendil.com/posts/you-said-no-mcp/', 'score': 191, 'story_id': 49906637}]}



Thought: I have identified the top story on Hacker News. Now, I need to fetch the most relevant comments for this story.

Using Tool: get_relevant_comments

Tool call dict: 
{'name': 'get_relevant_comments', 'arguments': {'story_id': 49906637, 'limit': 5}, 'id': 1}



Tool result: 
["This was the easiest call and many like me made it in March[0][1] among all of the anti-MCP wave of influencers claiming it dead (many, many prominent folks in tech including Garry Tan).  Literally every tech influencer in every social feed in March was calling MCP dead and crowning CLI the winner (completely ignoring every reasonable argument around security, ease of deployment and operations, etc.)<p>A direct quote from March, 2026:<p><pre><code>    &gt; If you\u2019re still not convinced that a lot of this discourse [regarding the death of MCP] lacks nuance and is just hype, congrats on buying into the current AI-influencer FOMO hype cycle; see you in 6 months when the influencers move on to the next revelation of the moment to stay relevant and get your eyeballs and dollars.\n</code></pre>\nIt was <i>fairly obvious</i> why MCP would be needed once AI engineering and uptake moved beyond the solo developer and single harness stack of &quot;what works for Me&quot; ver

The current top story on Hacker News is titled **[Pi.dev: You Said No MCP](https://earendil.com/posts/you-said-no-mcp/)** with a score of 191. Here are some of the first 5 comments about it:

1. This was the easiest call and many like me made it in March among all of the anti-MCP wave of influencers claiming it dead. It was fairly obvious why MCP would be needed once AI engineering and uptake moved beyond the solo developer. The key mistake was thinking in terms of their own workflows instead of a team's workflow.
   
2. I appreciate their reluctance towards MCP, but something is better than nothing. It’s suboptimal for the reasons the author outlines, but we use these huge technologies despite their flaws because they’re widely compatible.
   
3. I feel the same way about needing support for sub-agents. A smart model driving multiple dumber models will likely be a common pattern. I remember how much of a mess Eclipse was, which might be a sign of getting older.
   
4. I use mcporter for consuming MCP when suitable CLIs aren’t available. It exposes MCPs as shell commands, making it valuable when debugging.
   
5. I had no idea Pi didn't support MCP! I was just trying to get my tooling up and running and faced some pain in making connections.

If you need more information or assistance, feel free to ask!

The magenta **Thought** lines show the agent planning: it calls `get_hn_stories`, reads the `story_id` from the observation, then calls `get_relevant_comments` with it. That dependency is exactly what the single-round `ToolAgent` from `02_tool_use_pattern` could not handle.

---
## 💬 Part 5: A Streamlit Application

Finally, a chat UI. `app.py` keeps one agent per browser session and prefixes each question with the bot's earlier answers as context. Running this cell only writes the file; it doesn't need Streamlit installed.

In [8]:
%%writefile app.py
# ============================================================================
# STREAMLIT APP: a chat UI around the Hacker News bot
# ============================================================================
import streamlit as st
from dotenv import find_dotenv, load_dotenv

from hn_bot import get_hn_bot

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

# Set Streamlit page config
st.set_page_config(page_title="Hacker News Bot 🤖📰")
st.title("Hacker News Bot 🤖📰")

with st.sidebar:
    st.markdown("""
    # **Greetings, Digital Explorer!**

    Are you fatigued from navigating the expansive digital realm in search of your daily tech tales
    and hacker happenings? Fear not, for your cyber-savvy companion has descended upon the scene –
    behold the extraordinary **Hacker News Bot**!
    """)

# Initialize session state: one agent and the chat history per browser session
if "agent" not in st.session_state:
    st.session_state["agent"] = get_hn_bot(model="gpt-4o-mini")
if "messages" not in st.session_state:
    st.session_state["messages"] = []


def generate_response(question):
    """Generate response while passing conversation history as context."""
    context = "\n".join([msg["bot"] for msg in st.session_state["messages"]])
    response = st.session_state["agent"].run(f"Context: {context} Question: {question}")
    return response


# Display chat history
for msg in st.session_state["messages"]:
    st.chat_message("human").write(msg["user"])
    st.chat_message("ai").write(msg["bot"])

# Chat input handling
if prompt := st.chat_input():
    st.chat_message("human").write(prompt)
    with st.spinner("Thinking ..."):
        response = generate_response(prompt)
        st.chat_message("ai").write(response)

    # Store conversation in history
    st.session_state["messages"].append({"user": prompt, "bot": response})

Overwriting app.py


### Launch the app (by hand)

The UI is interactive, so the automated notebook run stops here. From the repository root:

```bash
uv sync --group apps
cd modules/01_agents_from_scratch/notebooks
uv run streamlit run app.py
```

Streamlit prints a local URL (usually http://localhost:8501). Ask it for the top stories, or for the comments on one of them. `hn_tools.py`, `hn_bot.py` and `app.py` are generated by this notebook and git-ignored.

---
## 📝 Summary

In this notebook, we learned:

### 1. The Architecture
- One ReAct agent, three tools, and the model decides which to call and in what order

### 2. Packaging the Agent Code
- The inline classes from 01–03 become importable modules (`agentic_patterns`), which is what lets an app reuse them

### 3. Real Tools
- A real API wrapped with `@tool`; helper functions stay plain Python
- Tools that return web content need size limits, or one call can flood the context

### 4. Running the Bot
- The ReAct loop chains dependent calls (story id → comments) without any code specific to that question

### 5. A Streamlit UI
- The same `get_hn_bot()` powers the notebook and the app; launch it with `uv run streamlit run app.py` after `uv sync --group apps`

### Next Steps
- **`01_defining_tools`** (module `02_tools_and_function_calling`): replace the prompt-and-parse tool plumbing with LangChain tools and the model's native tool calling